# Vehicle dynamics

Study the forward-only kinematic bicycle and actuator limits.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Parameters

The state pose is at the rear axle. Lengths are meters and time is seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.simulation.dynamics import (
    Control,
    VehicleParameters,
    VehicleState,
    bicycle_step,
    stopping_distance,
)

parameters = VehicleParameters()
dt = 0.1
print(parameters)

## Straight acceleration

A fixed acceleration reproduces the constant-acceleration distance equation until a bound is reached.

In [ ]:
state = VehicleState(speed=2)
trajectory = [state]
for _ in range(50):
    state = bicycle_step(state, Control(1, 0), dt, parameters)
    trajectory.append(state)
times = np.array([state.time for state in trajectory])
positions = np.array([state.pose.x for state in trajectory])
expected = 2 * times + 0.5 * times**2
np.testing.assert_allclose(positions, expected, atol=1e-10)
fig, ax = plt.subplots()
ax.plot(times, positions, label="Integrated")
ax.plot(times, expected, "--", label="Analytical")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Position (m)")
ax.legend()
plt.show()

## Steering response

The requested angle cannot change faster than the configured steering rate.

In [ ]:
state = VehicleState(speed=5)
turning = [state]
for _ in range(80):
    state = bicycle_step(state, Control(0, 0.35), dt, parameters)
    turning.append(state)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot([s.pose.x for s in turning], [s.pose.y for s in turning])
axes[0].set_aspect("equal")
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")
axes[1].plot([s.time for s in turning], [s.steering for s in turning])
axes[1].axhline(0.35, linestyle="--")
axes[1].set_ylabel("Steering (rad)")
axes[1].set_xlabel("Time (s)")
plt.show()

## Braking and no reverse

The model clamps speed at zero. Integration is approximate during the final partial stopping step.

In [ ]:
state = VehicleState(speed=10)
braking = [state]
for _ in range(40):
    state = bicycle_step(state, Control(-5, 0), dt, parameters)
    braking.append(state)
assert min(state.speed for state in braking) == 0
print("Analytical stopping distance:", stopping_distance(10, 5))
print("Discrete stopping distance:", braking[-1].pose.x)
fig, ax = plt.subplots()
ax.plot([s.time for s in braking], [s.speed for s in braking])
ax.set_xlabel("Time (s)")
ax.set_ylabel("Speed (m/s)")
plt.show()

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.